# Mock Pipeline Test Runner
Smoke-tests the full walk-forward pipeline end-to-end using the mock Synthea profile (1000 patients, 3 years).  
No modelling — stops after one walk-forward month.  

**Cells 1–10:** Initial creation (Synthea → BQ raw → slim → dictionaries → helpers → index_stay)  
**Cell 11:** Compute watermark from MAX(stop) − 1 year  
**Cell 12:** Walk-forward: one month via WalkForwardOrchestrator  
**Cell 13:** Final status check  

In [ ]:
# Cell 0: Setup & imports
import sys
import json
import shutil
import calendar
from datetime import date
from pathlib import Path

import pandas as pd

# Notebook lives at scripts/ — project root is one level up
project_root = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
# pipeline modules use both `from pipeline.x import` AND `from src.utils.x import`
sys.path.insert(0, str(project_root / "src"))
sys.path.insert(0, str(project_root))

from pipeline.synthea_runner import SyntheaRunner
from pipeline.synthea_segmenter import SyntheaSegmenter
from pipeline.bq_loader import BigQueryLoader
from pipeline.bq_transformer import BigQueryTransformer
from pipeline.dictionary_builder import DictionaryBuilder
from pipeline.walk_forward import WalkForwardOrchestrator

# Shared config paths (used across all cells)
synthea_config_path = str(project_root / "config" / "synthea_config.json")
config_path_bq     = str(project_root / "config" / "bigquery_config.json")
recipe_path        = str(project_root / "config" / "bigquery_recipes.json")
io_config_path     = str(project_root / "config" / "dictionary_io_config.json")
watermark_path     = str(project_root / "config" / "watermark.json")
checks_dir         = str(project_root / "sql" / "checks")

print(f"project_root: {project_root}")

In [ ]:
# Cell 1: Archive existing local files before overwriting
today = date.today().strftime("%Y%m%d")
archive_root = project_root / "data" / "archive" / today

FILES_TO_ARCHIVE = [
    # Synthea raw CSVs
    "data/raw/Synthea/mock/patients.csv",
    "data/raw/Synthea/mock/encounters.csv",
    "data/raw/Synthea/mock/careplans.csv",
    "data/raw/Synthea/mock/claims.csv",
    "data/raw/Synthea/mock/conditions.csv",
    "data/raw/Synthea/mock/medications.csv",
    "data/raw/Synthea/mock/organizations.csv",
    "data/raw/Synthea/mock/procedures.csv",
    # BQ fetch caches (data_path in dictionary_io_config.json)
    "data/raw/dictionaries/unique_diagnoses.csv",
    "data/raw/dictionaries/unique_procedures.csv",
    "data/raw/diagnoses_per_stays.csv",
    "data/raw/diagnoses_and_careplans.csv",
    "data/raw/diagnoses_and_following.csv",
    # SNOMED classification state
    "data/intermediate/diagnosess_snomed_state.json",
    "data/intermediate/procedures_snomed_state.json",
    # Processed dictionaries (write_path in dictionary_io_config.json)
    "data/processed/dictionaries/diagnoses_dictionary.csv",
    "data/processed/dictionaries/procedures_dictionary.csv",
    # Watermark
    "config/watermark.json",
]

archived, skipped = [], []
for rel in FILES_TO_ARCHIVE:
    src = project_root / rel
    if src.exists():
        dst = archive_root / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        archived.append(rel)
    else:
        skipped.append(rel)

print(f"Archive path: {archive_root}")
print(f"Archived ({len(archived)}):")
for f in archived:
    print(f"  {f}")
if skipped:
    print(f"\nNot found / skipped ({len(skipped)}):")
    for f in skipped:
        print(f"  {f}")

In [ ]:
# Cell 2: Run Synthea (mock: 1000 patients, 3 years, seed=100)
runner, run_params = SyntheaRunner.from_profile(synthea_config_path, "mock")
runner.run(**run_params)
print("Synthea done.")

In [ ]:
# Cell 2b: Segment Synthea CSVs into base + monthly files
segmenter = SyntheaSegmenter.from_profile(config_path_bq, "mock")
segmenter.segment(overwrite=True)

print(f"Segmentation complete.")
print(f"  simulation_start : {segmenter.simulation_start}")
print(f"  base_cutoff_date : {segmenter.base_cutoff_date}")

In [ ]:
# Cell 3: Load raw Synthea CSVs into BigQuery raw_data dataset
bq_loader, profile_cfg = BigQueryLoader.from_profile(config_path_bq, "mock")
bq_loader.load_profile_tables(profile_cfg)
print("Raw CSVs loaded to BQ.")

In [ ]:
# Cell 4: Create slim tables (recipe index 0)
transformer, _ = BigQueryTransformer.from_profile(config_path_bq)
transformer.run_query_sequence(recipe_path, 0, str(project_root))
print("Slim tables created.")

In [ ]:
# Cell 5: Build dictionaries locally via DictionaryBuilder
# New interface: no config_path_bq / profile_name args
builder = DictionaryBuilder(transformer=transformer, io_config_path=io_config_path)

builder.build_diagnoses_dictionary()
builder.build_procedures_dictionary()
builder.build_main_diagnoses()
print("Dictionaries built locally.")

In [ ]:
# Cell 6: Load dictionaries to BQ with plain table names (no mock_ prefix)
# Cannot use bq_loader.load_dictionaries() — it filters by profile_name in filename
# and adds mock_ prefix. The walk-forward update_* methods append to plain names
# (e.g. helper_tables.diagnoses_dictionary), so creation must match.
import json as _json
with open(config_path_bq) as f:
    _bq_cfg = _json.load(f)
helpers_dataset = _bq_cfg["dataset_helpers"]

helpers_loader = bq_loader.with_dataset(helpers_dataset)
helpers_loader.ensure_dataset_exists()

dict_dir = project_root / "data" / "processed" / "dictionaries"
dict_tables = {
    "diagnoses_dictionary.csv": "diagnoses_dictionary",
    "procedures_dictionary.csv": "procedures_dictionary",
    "main_diagnoses.csv": "main_diagnoses",
}

for filename, table_name in dict_tables.items():
    helpers_loader.load_one_csv(dict_dir / filename, table_name)

print("Dictionaries loaded to BQ.")

In [ ]:
# Cell 7: Build careplans_related_diagnoses locally + load to BQ
builder.build_careplans_related_diagnoses()

careplans_csv = project_root / "data" / "processed" / "careplans" / "careplans_related_encounters.csv"
helpers_loader.load_one_csv(careplans_csv, "careplans_related_encounters")

print("Careplans loaded to BQ.")

In [ ]:
# Cell 8: Create helper tables (recipe index 1) + run all 3 sanity checks
transformer.run_query_sequence(recipe_path, 1, str(project_root))
print("Helper tables created.")

transformer.run_helper_clinical_sanity_checks(checks_dir)
transformer.run_helper_cost_sanity_checks(checks_dir)
transformer.run_helper_utilization_sanity_checks(checks_dir)
print("All sanity checks passed.")

In [ ]:
# Cell 10: Create index_stay (recipe index 2) + instantiate orchestrator
transformer.run_query_sequence(recipe_path, 2, str(project_root))
print("index_stay created.")

orch = WalkForwardOrchestrator(
    transformer=transformer,
    dict_builder=builder,
    loader=bq_loader,
    recipe_path=recipe_path,
    project_root=str(project_root),
    watermark_path=watermark_path,
)
print("WalkForwardOrchestrator ready.")

In [ ]:
# Cell 11: Write watermark — derived from SyntheaSegmenter by WalkForwardOrchestrator
orch.initialize_watermark(segmenter.simulation_start, segmenter.base_cutoff_date)
print("Watermark initialized.")

In [ ]:
# Cell 12: Walk-forward — run one month
processed_end_date = orch.run_next_month()
print(f"Walk-forward month complete: {processed_end_date}")

In [ ]:
# Cell 12: Walk-forward — run one month
orch = WalkForwardOrchestrator(
    transformer=transformer,
    dict_builder=builder,
    recipe_path=recipe_path,
    project_root=str(project_root),
    watermark_path=watermark_path,
)

processed_end_date = orch.run_next_month()
print(f"\nWalk-forward month complete: {processed_end_date}")

In [ ]:
# Cell 13: Final status check
with open(watermark_path, encoding="utf-8") as f:
    wm = json.load(f)

print("=== Final watermark ===")
print(f"  last_processed_date : {wm['last_processed_date']}")
print(f"  next_end_date       : {wm['next_end_date']}")
print()
print(f"Processed month      : {wm['last_processed_date']}")
print(f"Next run would cover : window ending {wm['next_end_date']}")